# Imports

In [21]:
import os
from typing import Callable, List, Tuple, Any
from itertools import combinations

# CSV Reader

In [22]:
class CSVReader:
    @classmethod
    def __extract(cls, line: str):
        elements = line.split(",")
        for i in range(len(elements)):
            element_i: str = elements[i].strip().lower()
            try:
                elements[i] = float(element_i)
            except ValueError:
                elements[i] = element_i
        return elements
    
    @classmethod
    def fromFile(cls, filename: str) -> "CSVReader":
        values = []
        with open(filename) as file:
            columns = tuple(cls.__extract(file.readline()))
            for index, line in enumerate(file):
                row = cls.__extract(line)
                assert len(row) == len(columns), (
                    f"Row {index} has {len(row)} values but expected {len(columns)}"
                )
                values.append(row)
        return cls(columns, values)


    def __init__(self, columns: tuple, values: list):
        self.columns = columns
        self.values = values
        
    def __str__(self):
        s = f"{self.columns}"
        for line in self.values:
            s += f"\n{line}"
        return s
    
    def __len__(self):
        return len(self.values)
    
    def getColumns(self) -> tuple:
        return self.columns
    
    def getValues(self) -> list:
        return self.values

    def map(self, f: Callable[[List], Any], in_place: bool = False) -> "CSVReader":
        if in_place:
            reader = self
            values = self.getValues()
        else:
            reader = self.copy()
            values = reader.getValues()

        for i in range(len(values)):
            values[i] = f(values[i])

        return reader

    def reduce(self, f: Callable[[List, Any], Any], init_value: Any) -> Any:
        value: Any = init_value
        for line in self.values:
            value = f(line, value)
        return value
    
    def union(self, other: "CSVReader") -> "CSVReader":
        assert len(self) == len(other), f"Readers have different sizes {len(self)} vs {len(other)}"
        columns = self.getColumns() + other.getColumns()
        values0 = self.getValues()
        values1 = other.getValues()
        size = len(self)
        newValues = [None] * size
        for i in range(size):
            l0, l1 = values0[i], values1[i]
            newValues[i] = [None] * (len(l0) + len(l1))
            k = 0
            for j in range(len(l0)):
                newValues[i][k] = l0[j]
                k += 1
            for j in range(len(l1)):
                newValues[i][k] = l1[j]
                k += 1
        return CSVReader(columns, newValues)

    def copy(self) -> "CSVReader":
        newValue = [None] * len(self.values)
        for j in range(len(self.values)):
            line = self.values[j]
            newValue[j] = [None] * len(line)
            for i in range(len(line)):
                newValue[j][i] = line[i]        
        return CSVReader(self.columns, newValue)
    

# Main code

In [23]:
def k_anonymity(dataset: CSVReader, indexes: list) -> int:
    def f(attributes: list, accumulator: dict) -> dict:
        group = [None] * len(indexes)
        k = 0
        for i in indexes:
            group[k] = attributes[i]
            k += 1
        h = hash(f"{group}")
        if accumulator.get(h, None) == None:
            accumulator[h] = 1
        else:
            accumulator[h] += 1
        return accumulator
    results: dict = dataset.reduce(f, {})
    return min(results.values())

def distance(dataset: CSVReader, protected_dataset: CSVReader) -> float:
    assert dataset.getColumns() == protected_dataset.getColumns(), \
        f"Attributes mismatch {dataset.getColumns()} vs {protected_dataset.getColumns()}"
    assert len(dataset) == len(protected_dataset), \
        f"Sizes mismatch {len(dataset)} vs {len(protected_dataset)}"
    size = len(dataset)
    columns = dataset.getColumns()
    values, protected_values = dataset.getValues(), protected_dataset.getValues()
    distance = 0
    for i in range(size):
        for j in range(len(columns)):
            # chol,location,age,gender,height,frame,waist
            if columns[j] == "chol":
                d = (values[i][j] - protected_values[i][j]) / 100
            elif columns[j] == "age":
                d = (values[i][j] - protected_values[i][j]) / 50
            elif columns[j] == "height":
                d = (values[i][j] - protected_values[i][j]) / 15
            elif columns[j] == "waist":
                d = (values[i][j] - protected_values[i][j]) / 20
            elif columns[j] == "frame":
                if values[i][j] == "medium" or protected_values[i][j] == "medium":
                    d = 1
                else:
                    d = 2
            else: #if columns[j] in ("location", "gender"):
                d = 1 if values[i][j] == protected_values[i][j] else 0
            
            distance += d
    return distance


SEP = "#############################################################"

def parse_k_anonimity(dataset: CSVReader, min: int = 0):
    columns = dataset.getColumns()
    for i in range(1, len(columns)+1):
        for combo in combinations(columns, i):
            indexes = [columns.index(col) for col in combo]
            k = k_anonymity(dataset, indexes)
            if k > min:
                print(f"Test combination : {combo}")
                print("k-anonimity :", k)

def generalize(reader: CSVReader, function, column: str):
    columns = reader.getColumns()
    index = columns.index(column)
    assert index >= 0, f"Column not found {column}"
    reader.map(lambda values: function(values, index), in_place=True)

def generalize_int(values: list, index: int):
    age: float = values[index]
    values[index] = 10 * (int(age) // 10)
    return values


In [24]:
directory = os.path.dirname(os.curdir)
diabetes1 = CSVReader.fromFile(os.path.join(directory, "diabetes1.csv"))
diabetes2 = CSVReader.fromFile(os.path.join(directory, "diabetes2.csv"))

protected_diabetes1 = diabetes1.copy()
protected_diabetes2 = diabetes2.copy()

In [25]:
print(SEP)
print("Test K anonymity for diabetes1")
parse_k_anonimity(diabetes1, 2)

print(SEP)
print("Test K anonymity for diabetes2")
parse_k_anonimity(diabetes2, 2)

all_diabetes = diabetes1.union(diabetes2)
print(SEP)
print("Test K anonymity for both diabetes1 and diabetes2")
parse_k_anonimity(all_diabetes, 2)

#############################################################
Test K anonymity for diabetes1
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('location', 'gender')
k-anonimity : 81
#############################################################
Test K anonymity for diabetes2
Test combination : ('frame',)
k-anonimity : 100
#############################################################
Test K anonymity for both diabetes1 and diabetes2
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('frame',)
k-anonimity : 100
Test combination : ('location', 'gender')
k-anonimity : 81
Test combination : ('location', 'frame')
k-anonimity : 38
Test combination : ('gender', 'frame')
k-anonimity : 34
Test combination : ('location', 'gender', 'frame')
k-anonimity : 10


In [26]:
generalize(protected_diabetes1, generalize_int, "chol")
generalize(protected_diabetes1, generalize_int, "age")

generalize(protected_diabetes2, generalize_int, "height")
generalize(protected_diabetes2, generalize_int, "waist")

print(SEP)
print("Test K anonymity for diabetes1")
parse_k_anonimity(protected_diabetes1, 2)

print(SEP)
print("Test K anonymity for diabetes2")
parse_k_anonimity(protected_diabetes2, 2)

protected_all_diabetes = protected_diabetes1.union(protected_diabetes2)
print(SEP)
print("Test K anonymity for both protected diabetes1 and protected diabetes2")
parse_k_anonimity(protected_all_diabetes, 2)

#############################################################
Test K anonymity for diabetes1
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('location', 'gender')
k-anonimity : 81
#############################################################
Test K anonymity for diabetes2
Test combination : ('height',)
k-anonimity : 14
Test combination : ('frame',)
k-anonimity : 100
Test combination : ('waist',)
k-anonimity : 13
#############################################################
Test K anonymity for both protected diabetes1 and protected diabetes2
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('height',)
k-anonimity : 14
Test combination : ('frame',)
k-anonimity : 100
Test combination : ('waist',)
k-anonimity : 13


Test combination : ('location', 'gender')
k-anonimity : 81
Test combination : ('location', 'height')
k-anonimity : 4
Test combination : ('location', 'frame')
k-anonimity : 38
Test combination : ('location', 'waist')
k-anonimity : 3
Test combination : ('gender', 'frame')
k-anonimity : 34
Test combination : ('gender', 'waist')
k-anonimity : 5
Test combination : ('location', 'gender', 'frame')
k-anonimity : 10


In [27]:
print("Distance for diabetes1:", distance(diabetes1, protected_diabetes1))
print("Distance for diabetes2:", distance(diabetes2, protected_diabetes2))
print("Distance for all diabetes:", distance(all_diabetes, protected_all_diabetes))

Distance for diabetes1: 817.619999999996
Distance for diabetes2: 784.7499999999995
Distance for all diabetes: 1602.3699999999928


Adding Differential Privacy would optimizes our suppressions and generalizations such that in general the adding or removing of a specific user doesn't change too much an output result (predefined could be anything : mean, variance, more complex things). For example, don't add a King Kong in an animal database his removal should be quite visible.